In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import os
import glob
import shutil

In [3]:
os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver", exist_ok=True)

In [4]:
bronze_path = "/content/data_lake/bronze/sales.csv"

!wget -q -O /content/data_lake/bronze/sales.csv \
https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv

print("Bronze dataset created at:")
print(bronze_path)

Bronze dataset created at:
/content/data_lake/bronze/sales.csv


In [5]:
spark = (
    SparkSession.builder
    .appName("DataMiningDataLakeAssignment")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


In [6]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(bronze_path)
)

print("Bronze data loaded.")

Bronze data loaded.


In [7]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [8]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [9]:
before_count = df.count()

print("Total records before preprocessing:", before_count)

Total records before preprocessing: 1000


In [10]:
distinct_count = df.dropDuplicates().count()
duplicate_count = before_count - distinct_count

print("Total records:", before_count)
print("Distinct records:", distinct_count)
print("Exact duplicate rows:", duplicate_count)

Total records: 1000
Distinct records: 990
Exact duplicate rows: 10


In [11]:
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    print(f"\n--- {c} ---")
    (
        df.select(F.lower(F.trim(F.col(c))).alias("normalized_value"))
        .groupBy("normalized_value")
        .count()
        .orderBy("normalized_value")
        .show(50, truncate=False)
    )


--- customer_name ---
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|NULL            |10   |
|aarav agarwal   |2    |
|aarav chopra    |3    |
|aarav iyer      |2    |
|aarav kulkarni  |5    |
|aarav kumar     |2    |
|aarav nair      |2    |
|aarav patel     |6    |
|aarav singh     |4    |
|aarav tripathi  |2    |
|abhinav gupta   |6    |
|abhinav kulkarni|3    |
|abhinav malhotra|4    |
|abhinav pandey  |2    |
|abhinav pillai  |6    |
|abhinav rao     |6    |
|abhinav yadav   |4    |
|aditi agarwal   |1    |
|aditi chopra    |6    |
|aditi kapoor    |2    |
|aditi mishra    |3    |
|aditi patel     |9    |
|aditi tripathi  |1    |
|aditya chopra   |5    |
|aditya malhotra |2    |
|aditya mehta    |2    |
|aditya naidu    |1    |
|aditya rao      |3    |
|akash jain      |2    |
|akash kulkarni  |8    |
|akash pillai    |1    |
|akash reddy     |3    |
|akash saxena    |2    |
|akash singh     |2    |
|akash yadav     |5    |
|aman bose       |3    |
|a

In [12]:
for c in text_columns:
    inconsistent = df.filter(
        (F.col(c).isNotNull()) &
        (
            (F.trim(F.col(c)) != F.col(c)) |
            (F.col(c) != F.initcap(F.lower(F.trim(F.col(c)))))
        )
    )

    print(f"{c}: {inconsistent.count()} potentially inconsistent rows")

customer_name: 3 potentially inconsistent rows
product: 48 potentially inconsistent rows
category: 6 potentially inconsistent rows
payment_method: 378 potentially inconsistent rows
city: 9 potentially inconsistent rows
state: 0 potentially inconsistent rows
order_status: 0 potentially inconsistent rows


In [13]:
quantity_numeric = F.col("quantity").cast("double")

print("Invalid quantity rows:")
df.filter(
    quantity_numeric.isNull() | (quantity_numeric <= 0)
).show(20, truncate=False)

Invalid quantity rows:
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100114  |C0252      |Keerthi Singh|Ceiling Fan           |Home Appliances|0       |2920      |15              |Net Banking     |Chennai   |Tamil Nadu    |2025-02-15|Delivered   |
|100143  |C0184      |Aman Joshi   |Football              |Sports         |0       |1030      |15              |Debit Card      |Coimbatore|Tamil Nadu    |2025-11-26|Delivered   |
|100198  |C0014      |Arjun Pillai |Hoodie                |Clothing       |-2

In [14]:
unit_price_numeric = F.col("unit_price").cast("double")

print("Invalid unit_price rows:")
df.filter(
    unit_price_numeric.isNull() | (unit_price_numeric <= 0)
).show(20, truncate=False)

Invalid unit_price rows:
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category   |quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100102  |C0032      |Anjali Kumar    |Atta 10kg             |Grocery    |7       |0         |40              |Credit Card   |Mumbai    |Maharashtra   |2025-03-02|Processing  |
|100170  |C0158      |Deepak Pandey   |Competitive Exam Guide|Books      |6       |0         |0               |Net Banking   |Pune      |Maharashtra   |2025-08-26|Shipped     |
|100235  |C0349      |Tanvi Reddy     |Power Bank            |Electronics|7       |-500   

In [15]:
discount_numeric = F.col("discount_percent").cast("double")

print("Invalid discount rows:")
df.filter(
    discount_numeric.isNull() |
    (discount_numeric < 0) |
    (discount_numeric > 100)
).show(20, truncate=False)

Invalid discount rows:
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|order_id|customer_id|customer_name  |product         |category       |quantity|unit_price|discount_percent|payment_method  |city      |state     |order_date|order_status|
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|100034  |C0192      |Divya Sharma   |Yoga Mat        |Sports         |7       |1720      |-5              |Cash on Delivery|Coimbatore|Tamil Nadu|2025-08-19|Delivered   |
|100140  |C0075      |Swathi Tripathi|Cooking Oil 5L  |Grocery        |4       |790       |-5              |Credit Card     |Hyderabad |Telangana |2025-10-01|Shipped     |
|100217  |C0214      |Neha Tripathi  |Football        |Sports         |1       |650       |-5              |UPI      

In [16]:
parsed_date = F.coalesce(
    F.expr("try_to_timestamp(trim(order_date), 'yyyy-MM-dd')"),
    F.expr("try_to_timestamp(trim(order_date), 'dd/MM/yyyy')"),
    F.expr("try_to_timestamp(trim(order_date), 'dd-MM-yyyy')"),
    F.expr("try_to_timestamp(trim(order_date), 'MM/dd/yyyy')")
)

print("Invalid/unparseable dates:")

df.filter(
    F.col("order_date").isNotNull() &
    (F.trim(F.col("order_date")) != "") &
    parsed_date.isNull()
).show(20, truncate=False)

Invalid/unparseable dates:
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name|product      |category |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|100097  |C0244      |Shreya Rao   |Sofa         |Furniture|4       |47430     |10              |Net Banking     |Hyderabad|Telangana    |31/02/2026|Returned    |
|100358  |C0132      |Vikram Menon |Bedside Table|Furniture|7       |4460      |15              |Net Banking     |Lucknow  |Uttar Pradesh|2025/07/14|Delivered   |
|100448  |C0270      |Pranav Iyer  |Bedside Table|Furniture|1       |4290      |35              |UPI             |Lucknow  |Uttar Pradesh|not-a-date|Delivered

In [17]:
silver_df = df

before_count = df.count()
distinct_count = df.dropDuplicates().count()
duplicate_count = before_count - distinct_count

print("Rows before preprocessing:", before_count)
print("Exact duplicate rows:", duplicate_count)

Rows before preprocessing: 1000
Exact duplicate rows: 10


In [18]:
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

In [19]:
for c in text_columns:
    silver_df = silver_df.withColumn(
        c,
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            F.lit(None)
        ).otherwise(
            F.trim(F.col(c))
        )
    )

In [20]:
for c in ["customer_name", "product", "category", "city", "order_status"]:
    silver_df = silver_df.withColumn(
        c,
        F.when(
            F.col(c).isNotNull(),
            F.initcap(F.lower(F.col(c)))
        ).otherwise(F.col(c))
    )

In [21]:
silver_df = silver_df.withColumn(
    "state",
    F.when(
        F.col("state").isNotNull(),
        F.upper(F.col("state"))
    ).otherwise(F.col("state"))
)

In [22]:
silver_df = silver_df.withColumn(
    "payment_method",
    F.when(
        F.col("payment_method").isNull(),
        F.lit(None)
    ).when(
        F.lower(F.col("payment_method")) == "upi",
        F.lit("UPI")
    ).when(
        F.lower(F.col("payment_method")) == "cash",
        F.lit("Cash")
    ).when(
        F.lower(F.col("payment_method")) == "credit card",
        F.lit("Credit Card")
    ).when(
        F.lower(F.col("payment_method")) == "debit card",
        F.lit("Debit Card")
    ).otherwise(
        F.initcap(F.lower(F.col("payment_method")))
    )
)

In [23]:
for c in text_columns:
    silver_df = silver_df.fillna({c: "Unknown"})

In [24]:
silver_df = (
    silver_df
    .withColumn("quantity", F.col("quantity").cast("double"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount_percent", F.col("discount_percent").cast("double"))
)

silver_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = false)



In [25]:
silver_df = silver_df.withColumn(
    "quantity",
    F.when(
        F.col("quantity") > 0,
        F.col("quantity")
    ).otherwise(
        F.lit(None)
    )
)

In [26]:
silver_df = silver_df.withColumn(
    "unit_price",
    F.when(
        F.col("unit_price") > 0,
        F.col("unit_price")
    ).otherwise(
        F.lit(None)
    )
)

In [27]:
silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") >= 0) &
        (F.col("discount_percent") <= 100),
        F.col("discount_percent")
    ).otherwise(
        F.lit(None)
    )
)

In [28]:
quantity_median = silver_df.approxQuantile(
    "quantity",
    [0.5],
    0.01
)[0]

unit_price_median = silver_df.approxQuantile(
    "unit_price",
    [0.5],
    0.01
)[0]

discount_median = silver_df.approxQuantile(
    "discount_percent",
    [0.5],
    0.01
)[0]

print("Quantity median:", quantity_median)
print("Unit price median:", unit_price_median)
print("Discount median:", discount_median)

Quantity median: 5.0
Unit price median: 1780.0
Discount median: 20.0


In [29]:
silver_df = silver_df.fillna({
    "quantity": round(quantity_median),
    "unit_price": unit_price_median,
    "discount_percent": discount_median
})

In [30]:
silver_df = silver_df.withColumn(
    "quantity",
    F.col("quantity").cast(IntegerType())
)

In [31]:
silver_df = silver_df.withColumn(
    "parsed_order_date",
    F.coalesce(
        F.expr("try_to_timestamp(trim(order_date), 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(trim(order_date), 'dd/MM/yyyy')"),
        F.expr("try_to_timestamp(trim(order_date), 'dd-MM-yyyy')"),
        F.expr("try_to_timestamp(trim(order_date), 'MM/dd/yyyy')")
    )
)

In [32]:
silver_df = silver_df.withColumn(
    "order_date",
    F.date_format(
        F.col("parsed_order_date"),
        "yyyy-MM-dd"
    )
)

In [33]:
silver_df = silver_df.withColumn(
    "order_date",
    F.coalesce(
        F.col("order_date"),
        F.lit("Unknown")
    )
)

In [34]:
silver_df = silver_df.drop("parsed_order_date")

In [35]:
silver_df = silver_df.dropDuplicates()

In [36]:
silver_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = false)
 |-- discount_percent: double (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: string (nullable = false)
 |-- order_status: string (nullable = false)



In [37]:
silver_df.show(10, truncate=False)

+--------+-----------+------------------+-------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product            |category   |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+-------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100073  |C0066      |Pranav Patel      |Kurta              |Clothing   |3       |1970.0    |0.0             |Cash On Delivery|Hyderabad |TELANGANA     |2026-08-01|Delivered   |
|100125  |C0215      |Sai Naidu         |Sugar 5kg          |Grocery    |4       |410.0     |30.0            |Debit Card      |Hyderabad |TELANGANA     |2025-09-17|Returned    |
|100241  |C0343      |Pooja Jain        |Study Table        |Furniture  |10      |6570.0    |35.0            |

In [38]:
after_count = silver_df.count()

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing:", after_count)
print("Rows removed:", before_count - after_count)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed: 10


In [39]:
invalid_quantity_count = silver_df.filter(
    F.col("quantity") <= 0
).count()

invalid_unit_price_count = silver_df.filter(
    F.col("unit_price") <= 0
).count()

invalid_discount_count = silver_df.filter(
    (F.col("discount_percent") < 0) |
    (F.col("discount_percent") > 100)
).count()

print("Invalid quantity values:", invalid_quantity_count)
print("Invalid unit price values:", invalid_unit_price_count)
print("Invalid discount values:", invalid_discount_count)

Invalid quantity values: 0
Invalid unit price values: 0
Invalid discount values: 0


In [40]:
silver_null_counts = silver_df.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in silver_df.columns
])

silver_null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [41]:
silver_output_dir = "/content/data_lake/silver/preprocessed_sales"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_output_dir)

print("Silver data written successfully.")

Silver data written successfully.


In [42]:
part_files = glob.glob(
    f"{silver_output_dir}/part-*.csv"
)

print("Generated files:", part_files)

Generated files: ['/content/data_lake/silver/preprocessed_sales/part-00000-85da0a8b-4057-4e79-bf8d-95c1c6105aff-c000.csv']


In [43]:
final_silver_path = "/content/data_lake/silver/sales_silver.csv"

shutil.copy(
    part_files[0],
    final_silver_path
)

print("Final Silver file:")
print(final_silver_path)

Final Silver file:
/content/data_lake/silver/sales_silver.csv


In [44]:
silver_read = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(final_silver_path)
)

In [45]:
silver_read.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [46]:
silver_read.show(10, truncate=False)

+--------+-----------+------------------+-------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product            |category   |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+-------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100073  |C0066      |Pranav Patel      |Kurta              |Clothing   |3       |1970.0    |0.0             |Cash On Delivery|Hyderabad |TELANGANA     |2026-08-01|Delivered   |
|100125  |C0215      |Sai Naidu         |Sugar 5kg          |Grocery    |4       |410.0     |30.0            |Debit Card      |Hyderabad |TELANGANA     |2025-09-17|Returned    |
|100241  |C0343      |Pooja Jain        |Study Table        |Furniture  |10      |6570.0    |35.0            |

In [47]:
print("Silver row count:", silver_read.count())

Silver row count: 990


In [48]:
print("========== FINAL VERIFICATION ==========")
print("Bronze row count:", before_count)
print("Silver row count:", silver_read.count())
print("Exact duplicates originally:", duplicate_count)
print("Rows after preprocessing:", after_count)
print("Silver file exists:", os.path.exists(final_silver_path))
print("========================================")

========== FINAL VERIFICATION ==========
Bronze row count: 1000
Silver row count: 990
Exact duplicates originally: 10
Rows after preprocessing: 990
Silver file exists: True


## Preprocessing Summary

The Bronze layer contains the original sales.csv dataset without modification.

The Silver layer was created using Apache Spark with the following preprocessing operations:

1. Removed exact duplicate rows.
2. Handled missing values while preserving useful records.
3. Trimmed extra spaces from text fields.
4. Standardized obvious capitalization inconsistencies in text fields.
5. Validated quantity, unit price, and discount percentage.
6. Handled clearly invalid numerical values.
7. Converted valid dates to the consistent yyyy-MM-dd format.
8. Handled invalid or unparseable dates.
9. Saved the processed data as sales_silver.csv in the Silver layer.
10. Read the Silver output back using Spark and verified the row count and sample records.